In [1]:
# added

# source /data/software/xilinx/Vivado/2020.1/settings64.sh
# export XILINXD_LICENSE_FILE=2100@cselm2.ucsd.edu
# export LM_LICENSE_FILE=2100@cselm2.ucsd.edu

In [1]:
import os 
import pickle
import hashlib

import keras
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.losses import BinaryCrossentropy 
from sklearn.metrics import accuracy_score
import numpy as np

from qkeras.qlayers import QDense
from qkeras.quantizers import ternary

# os.environ['PATH'] = os.environ['XILINX_VIVADO'] + '/bin:' + os.environ['PATH'] # added: commented out for now
keras.utils.set_random_seed(32)

2026-02-05 09:59:53.257840: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-05 09:59:53.424218: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-02-05 09:59:53.424258: I tensorflow/compiler/xla/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-02-05 09:59:56.093838: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2026-

## Download data

In [2]:
# !wget https://zenodo.org/api/records/14427490/files-archive
# !unzip files-archive -d data
# !rm files-archive # optional cleanup 

# added: ran the above at command line to download and unzip data

## Setup data

In [12]:
# commented these two, data is in a directory above
train_data_dir = "./data"
test_data_dir = "./data"
# train_data_dir = "../data"
# test_data_dir = "../data"

start_location = 100
window_size = 400
end_window = start_location + window_size

# Added, max:
# start_location = 0
# window_size = 770
# end_window = start_location + window_size

### Added: What the data is:
Loading in 4 numpy arrays:
- X_train_val: (900000, T)
- y_train_val: (900000,)
- X_test shape: (100000, T)
- y_test shape: (100000,)

So, the data takes the form of (N, T) where T (770) is a long 1D sequence per N

The readout window is the subset of time samples from the qubit readout waveform that is chosen to keep and fed to the model.

Start location and end_window are indices in time, measured in samples (clock cycles).

Samples from start_location to end_window are kept, and multiply by 2 because the data is [I..., Q...].

Readout window = end_window - start_location.

Input size: 2 * readout_window.

Meaning: The qubit readout pulse lasts many clock cycles, early samples contain more noise(?). Later samples contain steady state resonator response, max separation between 0 and 1 states.

Overall, readout window is the part of the pulse that is trusted and used for classification.

Originally, Readout_window = 500 - 100, but this can be varied anywhere from 0 to 770 and a possible knob to optimize. So, the max Input size is then 1540.

In [13]:
"""Loadning training split"""
x_train_path = os.path.join(train_data_dir, f'0528_X_train_0_770.npy')
y_train_path = os.path.join(train_data_dir, f'0528_y_train_0_770.npy')

assert os.path.exists(x_train_path), f"ERROR: File {x_train_path} does not exist."
assert os.path.exists(y_train_path), f"ERROR: File {y_train_path} does not exist."

X_train_val = np.load(x_train_path)
y_train_val = np.load(y_train_path)

# Insure same dataset is loaded 
assert hashlib.md5(X_train_val).hexdigest() == 'b61226c86b7dee0201a9158455e08ffb',  "Checksum failed. Wrong file was loaded or file may be corrupted."
assert hashlib.md5(y_train_val).hexdigest() == 'c59ce37dc7c73d2d546e7ea180fa8d31',  "Checksum failed. Wrong file was loaded or file may be corrupted."

# Get readout window
X_train_val = X_train_val[:,start_location*2:end_window*2]
assert len(X_train_val[0]) == (end_window-start_location)*2, f"ERROR: X_test sample size {len(X_train_val[0])} does not match (start window, end window) ({start_location},{end_window}) size."

In [15]:
"""Loading testing split"""
x_test_path = os.path.join(test_data_dir, f'0528_X_test_0_770.npy')
y_test_path = os.path.join(test_data_dir, f'0528_y_test_0_770.npy')

assert os.path.exists(x_test_path), f"ERROR: File {x_test_path} does not exist."
assert os.path.exists(y_test_path), f"ERROR: File {y_test_path} does not exist."

X_test = np.load(x_test_path)
y_test = np.load(y_test_path)

# Insure same dataset is loaded 
assert hashlib.md5(X_test).hexdigest() == 'b7d85f42522a0a57e877422bc5947cde', "Checksum failed. Wrong file was loaded or file may be corrupted."
assert hashlib.md5(y_test).hexdigest() == '8c9cce1821372380371ade5f0ccfd4a2', "Checksum failed. Wrong file was loaded or file may be corrupted."

# Get readout window
X_test = X_test[:,start_location*2:end_window*2]
assert len(X_test[0]) == (end_window-start_location)*2, f"ERROR: X_test sample size {len(X_test[0])} does not match (start window, end window) ({start_location},{end_window}) size."

In [16]:
# added: print dataset shapes and class distribution

print(f"X_train_val shape: {X_train_val.shape}")
print(f"y_train_val shape: {y_train_val.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

T = X_train_val.shape[1] // 2   # before window slicing
print(T)

X_train_val shape: (900000, 1540)
y_train_val shape: (900000,)
X_test shape: (100000, 1540)
y_test shape: (100000,)
770


## Construct the model
 
QKeras is "Quantized Keras" for deep heterogeneous quantization of ML models. We're using QDense layer instead of Dense. We're also training with model sparsity, since QKeras layers are prunable.

In [7]:
model = keras.models.Sequential()
model.add(QDense(
    4, 
    activation=None, 
    name='fc1',
    input_shape=(800,), 
    kernel_quantizer=ternary(),
    bias_quantizer=ternary(),
))
model.add(BatchNormalization(name='batchnorm1'))
model.add(QDense(
    1, 
    name='fc2', 
    activation='sigmoid', 
    kernel_quantizer=ternary(),
    bias_quantizer=ternary(),
))

print(model.summary())

2026-01-28 09:55:02.121447: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2026-01-28 09:55:02.122180: W tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:265] failed call to cuInit: UNKNOWN ERROR (303)
2026-01-28 09:55:02.122199: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (mulder.t2.ucsd.edu): /proc/driver/nvidia/version does not exist
2026-01-28 09:55:02.122464: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 fc1 (QDense)                (None, 4)                 3204      
                                                                 
 batchnorm1 (BatchNormalizat  (None, 4)                16        
 ion)                                                            
                                                                 
 fc2 (QDense)                (None, 1)                 5         
                                                                 
Total params: 3,225
Trainable params: 3,217
Non-trainable params: 8
_________________________________________________________________
None


## Train the model

In [8]:
init_learning_rate = 1e-3
validation_split = 0.05  # 45,000 sample size 
batch_size = 256
epochs = 5 # 50
early_stopping_patience = 10

checkpoint_dir = f'checkpoints/'
checkpoint_filename = f'qkeras_ckp_model_best.h5'
ckp_filename = os.path.join(checkpoint_dir, checkpoint_filename)

if os.path.exists(checkpoint_dir) == False:
    print(f'Checkpoint directory {checkpoint_dir} does not exist.')
    print('Creating directory...')
    os.mkdir(checkpoint_dir)

Checkpoint directory checkpoints/ does not exist.
Creating directory...


In [9]:
# train = False # commented out
train = True # added to train

if train: 
    opt = Adam(learning_rate=init_learning_rate)
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=early_stopping_patience,
            restore_best_weights=True,
        ),
    ] 
    model.compile(
        optimizer=opt, 
        loss=BinaryCrossentropy(from_logits=False), 
        metrics=['accuracy']
    )
    history = model.fit(
        X_train_val, 
        y_train_val, 
        batch_size=batch_size,
        epochs=epochs, 
        validation_split=validation_split, 
        shuffle=True, 
        callbacks=callbacks,
    )
    
    model.save_weights(os.path.join(checkpoint_dir, 'qkeras_model_best.h5'))
    # Save the history dictionary
    with open(os.path.join(checkpoint_dir, f'qkeras_training_history.pkl'), 'wb') as f:
        pickle.dump(history.history, f)    
else: 
    model.load_weights(os.path.join(checkpoint_dir, checkpoint_filename))

Epoch 1/5
3340/3340 [==============================] - 7s 2ms/step - loss: 0.1959 - accuracy: 0.9558 - val_loss: 0.1782 - val_accuracy: 0.9581
Epoch 2/5
3340/3340 [==============================] - 6s 2ms/step - loss: 0.1801 - accuracy: 0.9592 - val_loss: 0.1770 - val_accuracy: 0.9583
Epoch 3/5
3340/3340 [==============================] - 6s 2ms/step - loss: 0.1786 - accuracy: 0.9594 - val_loss: 0.1782 - val_accuracy: 0.9583
Epoch 4/5
3340/3340 [==============================] - 6s 2ms/step - loss: 0.1784 - accuracy: 0.9597 - val_loss: 0.1778 - val_accuracy: 0.9585
Epoch 5/5
3340/3340 [==============================] - 6s 2ms/step - loss: 0.1792 - accuracy: 0.9594 - val_loss: 0.1769 - val_accuracy: 0.9582


## Check performance

In [10]:
# get ground and excited indices 
e_indices = np.where(y_test == 1)[0]
g_indices = np.where(y_test == 0)[0]

# separate ground and excited samples 
Xe_test = X_test[e_indices]
ye_test = y_test[e_indices]

Xg_test = X_test[g_indices]
yg_test = y_test[g_indices]

# compute total correct for excited state 
ye_pred = model.predict(Xe_test)
ye_pred = np.where(ye_pred < 0.5, 0, 1).reshape(-1)
e_accuracy = accuracy_score(ye_test, ye_pred)

total_correct = (ye_test==ye_pred).astype(np.int8).sum()
total_incorrect = (ye_test!=ye_pred).astype(np.int8).sum()

# compute total correct for ground state 
yg_pred = model.predict(Xg_test)
yg_pred = np.where(yg_pred < 0.5, 0, 1)
g_accuracy = accuracy_score(yg_test, yg_pred)

total_correct = (yg_test==yg_pred).astype(np.int8).sum()
total_incorrect = (yg_test!=yg_pred).astype(np.int8).sum()

# compute fidelity 
test_fidelity = 0.5*(e_accuracy + g_accuracy)
test_fidelity = test_fidelity*2-1
test_fidelity = 1/2 + (0.5*test_fidelity)

y_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, np.where(y_pred < 0.5, 0, 1).reshape(-1))

print('\n===================================')
print('    Accuracy', test_acc)
print('    Fidelity', test_fidelity)

3125/3125 [==============================] - 3s 953us/step

    Accuracy 0.95953
    Fidelity 0.95953
